In [1]:
import pandas as pd
import numpy as np

beneficiary = pd.read_csv(
    "../data/processed/beneficiary.csv"
)

inpatient = pd.read_csv(
    "../data/processed/inpatient.csv"
)

In [ ]:
datasets = [beneficiary, inpatient]

for dataset in datasets:
    dataset.dropna(subset=["desynpuf_id"], inplace=True)
    dataset.drop_duplicates(inplace=True)
    dt_columns = [col for col in dataset.columns if col.endswith('_dt')]
    for col in dt_columns:
        dataset[col] = pd.to_datetime(dataset[col], format='%Y%m%d', errors='coerce')

desynpuf_id                         object
bene_birth_dt               datetime64[ns]
bene_death_dt               datetime64[ns]
bene_sex_ident_cd                    int64
bene_race_cd                         int64
bene_esrd_ind                       object
sp_state_code                        int64
bene_county_cd                       int64
bene_hi_cvrage_tot_mons              int64
bene_smi_cvrage_tot_mons             int64
bene_hmo_cvrage_tot_mons             int64
plan_cvrg_mos_num                    int64
sp_alzhdmta                          int64
sp_chf                               int64
sp_chrnkidn                          int64
sp_cncr                              int64
sp_copd                              int64
sp_depressn                          int64
sp_diabetes                          int64
sp_ischmcht                          int64
sp_osteoprs                          int64
sp_ra_oa                             int64
sp_strketia                          int64
medreimb_ip

In [3]:
beneficiary = beneficiary[
    beneficiary["bene_birth_dt"] <= beneficiary["bene_death_dt"]
]

inpatient = inpatient[
    (inpatient["clm_admsn_dt"] <= inpatient["nch_bene_dschrg_dt"]) &
    (inpatient["clm_from_dt"] <= inpatient["clm_thru_dt"])
]

In [ ]:
df = inpatient.merge(
    beneficiary,
    on="desynpuf_id",
    how="left"
)

df = df.sort_values(
    by=["desynpuf_id", "clm_admsn_dt"]
)

df["previous_discharge"] = (
    df.groupby("desynpuf_id")["nch_bene_dschrg_dt"]
    .shift(1)
)

df["days_since_last_admission"] = (
    df["clm_admsn_dt"]
    - df["previous_discharge"]
).dt.days

df["next_admission"] = (
    df.groupby("desynpuf_id")["clm_admsn_dt"]
    .shift(-1)
)

df["days_until_next_admission"] = (
    df["next_admission"]
    - df["nch_bene_dschrg_dt"]
).dt.days

df["readmitted_30d"] = (
    df["days_until_next_admission"] <= 30
).astype(int)

desynpuf_id             object
clm_id                   int64
segment                  int64
clm_from_dt     datetime64[ns]
clm_thru_dt     datetime64[ns]
                     ...      
benres_op              float64
pppymt_op              float64
medreimb_car           float64
benres_car             float64
pppymt_car             float64
Length: 112, dtype: object


In [ ]:
df["icd9_dgns_cd_1"] = (
    df["icd9_dgns_cd_1"]
    .astype(str)
)

df["icd9_dgns_cd_1"].value_counts().head(20)

icd9_dgns_cd_1
486      2516
V5789    1801
41401    1689
0389     1659
49121    1596
4280     1497
5990     1461
42731    1242
41071    1171
5849     1137
71536    1097
43491    1030
51881     792
78659     748
7802      735
5070      724
27651     637
82021     593
4359      541
6826      502
Name: count, dtype: int64

In [7]:
df.isnull().sum().sort_values(ascending=False)

hcpcs_cd_13                       66961
hcpcs_cd_14                       66961
hcpcs_cd_15                       66961
hcpcs_cd_16                       66961
hcpcs_cd_17                       66961
                                  ...  
nch_bene_blood_ddctbl_lblty_am        0
clm_utlztn_day_cnt                    0
nch_bene_dschrg_dt                    0
clm_drg_cd                            0
readmitted_30d                        0
Length: 117, dtype: int64

In [ ]:
df["clm_utlztn_day_cnt"] = (
    df["clm_utlztn_day_cnt"]
    .fillna(df["clm_utlztn_day_cnt"].median())
)

df["icd9_dgns_cd_1"] = (
    df["icd9_dgns_cd_1"]
    .fillna("UNKNOWN")
)

In [ ]:
df["days_since_last_admission"] = (
    df["days_since_last_admission"]
    .fillna(-1)
)

df["length_of_stay"] = (
    df["nch_bene_dschrg_dt"]
    - df["clm_admsn_dt"]
).dt.days

df["prior_admissions"] = (
    df.groupby("desynpuf_id")
    .cumcount()
)

chronic_cols = [
    "sp_chf",
    "sp_diabetes",
    "sp_copd"
]

df["chronic_burden"] = (
    df[chronic_cols] == 1
).sum(axis=1)

In [ ]:
model_df = df[
    [
        "readmitted_30d",
        "length_of_stay",
        "prior_admissions",
        "days_since_last_admission",
        "chronic_burden",
        "bene_sex_ident_cd",
        "bene_race_cd",
        "clm_utlztn_day_cnt"
    ]
]

model_df.to_csv(
    "../data/processed/model_dataset.csv",
    index=False
)